# Field (KIE) detection fine-tuning on Colab

Trains the multi-class `db_resnet50` field detector from `references/detection` on a GPU runtime.

**Before running**: `Runtime > Change runtime type > GPU` (L4 or A100 recommended). Upload to Google Drive:

- `MyDrive/doctr_data/doctr.zip`: the zipped output of `convert_documentai.py` (folder `doctr/` with `train/`, `val/`, `test/`)
- optionally `MyDrive/doctr_data/<checkpoint>.pt` to resume from

Checkpoints are written to Drive so a disconnected session loses nothing: resume from the last `_epochN.pt`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = "finetune-invoice"
REPO = "https://github.com/fercho1999leon/doctr.git"
!git clone -q --branch $BRANCH $REPO /content/doctr
%cd /content/doctr
!pip install -q -e . tqdm matplotlib
import torch

print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
DRIVE = "/content/drive/MyDrive/doctr_data"
!mkdir -p /content/datasets $DRIVE/checkpoints
!unzip -q -o $DRIVE/doctr.zip -d /content/datasets
!ls /content/datasets/doctr && python -c "import json;print(len(json.load(open('/content/datasets/doctr/train/labels.json'))),'train images')"

## Train

Set `RESUME` to a checkpoint in Drive to continue a previous run (then `--pretrained` is not used), or leave it empty to start from the ImageNet/doctr pretrained weights. Batch size 8 fits an L4/A100; use 4 on a T4.

In [ ]:
RESUME = ""  # e.g. f'{DRIVE}/db_resnet50_fields_r2_epoch21.pt'
NAME = "db_resnet50_fields_colab"
BATCH = 8
EPOCHS = 60
LR = 3e-4 if RESUME else 1e-3
init = f"--resume {RESUME}" if RESUME else "--pretrained"
!python references/detection/train.py db_resnet50 {init} \
  --train_path /content/datasets/doctr/train --val_path /content/datasets/doctr/val \
  --device 0 --amp -b {BATCH} --epochs {EPOCHS} --lr {LR} --sched cosine --no-hflip --exhaustive-labels \
  --early-stop --early-stop-epochs 15 --early-stop-delta 0.001 --save-interval-epoch \
  --output_dir {DRIVE}/checkpoints --name {NAME} -j 4

## Evaluate per field

First on `val` (tune thresholds here), then once on `test` for the final number.

In [ ]:
!python references/detection/evaluate_fields.py --checkpoint {DRIVE}/checkpoints/{NAME}.pt --data /content/datasets/doctr/val --top-k-per-class 1

In [ ]:
!python references/detection/evaluate_fields.py --checkpoint {DRIVE}/checkpoints/{NAME}.pt --data /content/datasets/doctr/test --top-k-per-class 1

## Try it on one image

In [ ]:
!python references/detection/kie_inference.py --checkpoint {DRIVE}/checkpoints/{NAME}.pt --top-k-per-class 1 $(ls /content/datasets/doctr/test/images/* | head -1)

When you are done: `Runtime > Disconnect and delete runtime` so the session stops consuming compute units.